In [ ]:
import json
import pandas as pd
import torch
import numpy as np

In [ ]:
slide_lenght = 512 #tokens
slide_overlap = 128 #tokens
text_max_lenght = 10000 #characters

In [ ]:
from transformers import pipeline, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "openai/privacy-filter",
    model_max_length=slide_lenght,
    truncation=True,
)

classifier = pipeline(
    task="token-classification",
    model="openai/privacy-filter",
    tokenizer=tokenizer,
    device=0,
)

classifier("My name is Alice Smith")


config.json:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.80G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

[{'entity': 'B-private_person',
  'score': np.float32(0.9999994),
  'index': 3,
  'word': 'ĠAlice',
  'start': 10,
  'end': 16},
 {'entity': 'E-private_person',
  'score': np.float32(0.9999919),
  'index': 4,
  'word': 'ĠSmith',
  'start': 16,
  'end': 22}]

In [ ]:
data_names = [
    "C4",
    "finepdfs",
    "fineweb",
    "RedPajama",]

In [ ]:
def merge_tags(private_tags):
    new_tags = []
    inside = False # are we inside of private token sequenze?
    for tag in private_tags:
        label = tag["entity"]
        label_loc = label[0]
        label_kind = label[2:]
        if label_loc == 'S':
            new_tag = {"entity": label_kind,
                       "word": tag["word"],
                       "start": tag["start"],
                       "end": tag["end"]}
            new_tags.append(new_tag)
            inside = False
        elif label_loc == 'B':
            new_tag = {"entity": label_kind,
                       "word": tag["word"],
                       "start": tag["start"],
                       "end": tag["end"]}
            inside = True
        elif label_loc == 'I' and inside:
            new_tag["word"] += tag["word"]
        elif label_loc == 'E' and inside:
            new_tag["word"] += tag["word"]
            new_tag["end"] = tag["end"]
            new_tags.append(new_tag)
            inside = False
    for tag in new_tags:
        # Replace the special RoBERTa space with a real space, and strip edges
        tag["word"] = tag["word"].replace("Ġ", " ").strip()
    return new_tags

In [ ]:
def convert_floats(obj):
    if isinstance(obj, np.float32):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: convert_floats(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_floats(elem) for elem in obj]
    return obj

In [ ]:
analysis = {}
for name in data_names:
    with open(f"{name}_dataset.json", 'r') as f:
        data = json.load(f)
    print(f"Opened {name}")
    text_key = 'raw_content' if name == 'RedPajama' else 'text'
    texts = [datapoint[text_key] for datapoint in data]
    texts_truncated = [text[:text_max_lenght] for text in texts]
    analysis[name] = classifier(
        texts_truncated,
        batch_size=1,
        stride=slide_overlap
    )
    print(f"{name} analysis done")
    analysis_merged = []
    for i in range(30):
        analysis_merged.append(merge_tags(analysis[name][i]))
    analysis[name] = analysis_merged
    print(f"tags merged")
    analysis_serializable = convert_floats(analysis[name]) # because else we get troubles with floats when trying to save as json
    with open(f"{name}_analysis.json", 'w') as f:
        json.dump(analysis_serializable, f, indent=4)
    print(f"{name} done.\n")

Opened RedPajama
RedPajama analysis done
tags merged
RedPajama done.

